# Import needed functions

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import kurtosis
from statsmodels.tsa.stattools import acf
import pickle as pkl
import time
from joblib import Parallel, delayed

# Define LOB functions

In [ ]:
def sort_book(bids, asks):
    # bids: higher price first, then earlier tplaced
    bids.sort(key=lambda o: (-o["price"], o["t_placed"]))
    # asks: lower price first, then earlier tplaced
    asks.sort(key=lambda o: (o["price"], o["t_placed"]))
    return bids, asks

def best_bid(bids):
    return bids[0]["price"] if bids else np.nan

def best_ask(asks):
    return asks[0]["price"] if asks else np.nan

def mid_price(bids, asks):
    if asks and bids:
        bb = best_bid(bids)
        ba = best_ask(asks)
        return 0.5 * (bb + ba)
    else:
        return np.nan

def forward_fill(M):
    '''This function ensures that whenever the bids or asks is empty, the M array take the previous value.'''
    mask = np.isnan(M)
    M[mask] = np.interp(np.flatnonzero(mask), np.flatnonzero(~mask), M[~mask])
    return M

def warm_LOB(parameters):
    # Define a warm LOB
    bids = [{'price': parameters['P_f']-i*parameters['delta'],
            'qty': parameters['D'],
            't_placed': 0} for i in range(1, parameters['L']+1)]

    asks = [{'price': parameters['P_f']+i*parameters['delta'],
            'qty': parameters['D'],
            't_placed': 0} for i in range(1, parameters['L']+1)]

    bids, asks = sort_book(bids, asks)

    return bids, asks


def execute_match(bids, asks):
    """Remove any empty qty orders. Keep books sorted."""
    bids[:] = [o for o in bids if o["qty"] > 0]
    asks[:] = [o for o in asks if o["qty"] > 0]
    return sort_book(bids, asks)

def place_and_match(order, t, bids, asks):
    """
    Place a single order of qty 1 at given price.
    If it crosses the book, execute one unit against best opposite side.
    Otherwise add it to the book.
    """
    price = order["price"]
    qty = 1

    # Buyer crossing condition: price >= best ask
    # Seller crossing condition: price <= best bid
    if order["side"] == "buy":
        if asks and price >= asks[0]["price"]:
            # Trade one unit at current best ask
            asks[0]["qty"] -= 1
            # asks[0]['t_placed'] = asks[0]['t_placed'].remove(min(asks[0]['t_placed'])) # FIFO Rule
            bids, asks = execute_match(bids, asks)
            return bids, asks ,{"traded": True}
        else:
            # Add limit buy to bids
            bids.append({"price": price, "qty": qty, "t_placed": t})
            bids, asks = execute_match(bids, asks)
            return bids, asks ,{"traded": False}
    else:
        # side == "sell"
        if bids and price <= bids[0]["price"]:
            # Trade one unit at current best bid
            bids[0]["qty"] -= 1
            # asks[0]['t_placed'] = asks[0]['t_placed'].remove(min(asks[0]['t_placed'])) # FIFO Rule
            bids, asks = execute_match(bids, asks)
            return bids, asks ,{"traded": True}
        else:
            # Add limit sell to asks
            asks.append({"price": price, "qty": qty, "t_placed": t})
            bids, asks = execute_match(bids, asks)
            return bids, asks ,{"traded": False}

def remove_exp_orders(asks, bids, t, parameters):
    '''This function removes all of expired orders after t_c'''
    t_exp = t - parameters['t_c']
    # if t_exp >= 0:  
    #     for k ,ask in enumerate(asks):
    #         if any(i <= t_exp for i in ask['t_placed']):
    #             count = sum(1 for i in ask['t_placed'] if i <= t_exp)
    #             ask['qty'] -= count
    #             ask['t_placed'] = [j for j in ask['t_placed'] if j > t_exp]
    #             asks[k] = ask

    #     for k ,bid in enumerate(bids):
    #         if any(i <= t_exp for i in bid['t_placed']):
    #             count = sum(1 for i in bid['t_placed'] if i <= t_exp)
    #             bid['qty'] -= count
    #             bid['t_placed'] = [j for j in bid['t_placed'] if j > t_exp]
    #             bids[k] = bid

    if t_exp >= 0:
        asks = [ask for ask in asks if ask['t_placed'] > t_exp]
        bids = [bid for bid in bids if bid['t_placed'] > t_exp]

    return asks, bids


def calcualte_outputs(parameters, M):
    # Define K for obtaining non-overlaping 100-tick return
    K = [i for i in range(1, 1+int((parameters['T_E']-parameters['B'])/100))]

    # Calculate returns
    r = np.ones(len(K))
    for k in K:
        r[k-1] = np.log(M[100*k] / M[100*(k-1)])

    # Calcualte excess kurtosis
    excess_kurt_biased = kurtosis(r, bias=True)
    excess_kurt_unbiased = kurtosis(r, bias=False)

    # Calcualte acf
    max_lag = 6
    acf_sq_vals = acf(r**2, nlags=max_lag)

    output = {f'acf_lag{i}': j for i,j in enumerate(acf_sq_vals) if i!=0}
    output['Excess_kurtosis'] =  excess_kurt_unbiased

    return output

def single_run(parameters, run):
    seed = 1234 + run
    M = Run(parameters, seed)
    output = calcualte_outputs(parameters, M)
    return output

def simulation_parallel(parameters, n_workers=4):
    R = parameters["R"]

    outputs = Parallel(n_jobs=n_workers)(
        delayed(single_run)(parameters, run)
        for run in range(R)
    )

    metrics = {f"acf_lag{i}": [] for i in range(1, 7)}
    metrics["Excess_kurtosis"] = []

    for output in outputs:
        for i in range(1, 7):
            metrics[f"acf_lag{i}"].append(output[f"acf_lag{i}"])
        metrics["Excess_kurtosis"].append(output["Excess_kurtosis"])

    for i in range(1, 7):
        metrics[f"acf_lag{i}"] = np.mean(metrics[f"acf_lag{i}"])

    metrics["Excess_kurtosis"] = np.mean(metrics["Excess_kurtosis"])

    return metrics

# Zero-intelligence traders

In [ ]:
def assign_traders(parameters):
    '''
    This function assign sellers and buyyers traders to trader list 
    and return that.
    '''
    
    N = parameters['N']
    P_f = parameters['P_f']
    S = parameters['S']
    

    buy_values = np.random.uniform(P_f, P_f+S, size=int(N/2))
    sell_values = np.random.uniform(P_f-S, P_f, size=int(N/2))

    buyers = [{'type': 'buyer',
                'v_or_c': i} for i in buy_values]
    sellers = [{'type': 'seller',
                'v_or_c': i} for i in sell_values]
    traders = []

    for i in range(N):
        n = np.random.uniform(0,1)
        if n >= 0.5:
            if len(buyers) == 0:
                traders.append(sellers.pop(0))
            else:
                traders.append(buyers.pop(0))
        else:
            if len(sellers)== 0:
                traders.append(buyers.pop(0))
            else:
                traders.append(sellers.pop(0))

    return traders


def Run(parameters, seed):
    np.random.seed(seed)
    
    # Initiate the LOB
    bids, asks = warm_LOB(parameters)

    # Initiate traders
    traders = assign_traders(parameters)

    M = []  # mid price path
    M.append(mid_price(bids, asks))  # M0

    # Start the simulation
    for t in range(1, parameters['T_E']+1):
        
        asks, bids = remove_exp_orders(asks, bids, t, parameters)
        bids, asks = execute_match(bids, asks)
        tr = traders[(t-1) % parameters['N']]
        if tr["type"] == "buyer":
            price = np.random.uniform(parameters['P_min'], tr["v_or_c"])
            price = int(np.floor(price))
            new_order = {"side": "buy", "price": price}
        else:
            price = np.random.uniform(tr["v_or_c"], parameters['P_max'])
            price = int(np.ceil(price))
            new_order = {"side": "sell", "price": price}
        bids, asks, traded= place_and_match(new_order, t, bids, asks)
        M.append(mid_price(bids, asks))

    M = M[parameters['B']:] # Burn-in 
    M = np.array(M, dtype=float)
    M = forward_fill(M) # Keep the previous value if current value is nan(Bids or Asks is empty)
    return M



In [ ]:
# Define parameters
parameters1 = {
    'N' : 500,
    'P_f' : 10_000,
    'S' : 1_000,
    'delta_P' : 1,
    't_c' : 10_000,
    'T_E' : 100_000,
    'L' : 200,
    'delta' : 1,
    'D' : 5,
    'R' : 30
}
parameters1['P_min'] = parameters1['delta_P']
parameters1['P_max'] = 2 * (parameters1['P_f'] + parameters1['S'])
parameters1['B'] = int(0.01 * parameters1['T_E'])

metrics1 = simulation_parallel(parameters1, n_workers=40)

with open(f"metrics_1.pkl", "wb") as f:
    pkl.dump(metrics1, f)

In [2]:
with open("metrics_1.pkl", "rb") as f:
    metrics1 = pkl.load(f)
print(f'ZIT Metrics:\n{metrics1}')

ZIT Metrics:
{'acf_lag1': np.float64(0.19694755833802147), 'acf_lag2': np.float64(0.08194747940461908), 'acf_lag3': np.float64(0.07199381686045846), 'acf_lag4': np.float64(0.07382788068181118), 'acf_lag5': np.float64(0.07709059279298024), 'acf_lag6': np.float64(0.05378217311976083), 'Excess_kurtosis': np.float64(1.0437044940789921)}


# Behavioural traders

In [5]:
def assign_traders_weights(parameters):
    '''
    This function assign traders' weights.
    '''
    N = parameters['N']
    
    traders = [i for i in range(N)]
    for i in range(N):
        traders[i] = {}
        traders[i]['w1'] = np.random.uniform(0, parameters['w1_max'])
        traders[i]['w2'] = np.random.uniform(0, parameters['w2_max'])
        traders[i]['w3'] = np.random.uniform(0, parameters['w3_max'])
        traders[i]['tau'] = int(np.random.uniform(1, parameters['tau_max']))

    return traders

def traders_price(parameters, trw, M):
    tau = trw['tau']

    if len(M) >= tau+1:
        x = M[-1-tau]
    else:
        x = M[0]

    eps = np.random.normal(0, parameters['sigma_noise'])
    nom = (trw['w1']*np.log(parameters['P_f']/M[-1]) + trw['w2']*np.log(M[-1]/x) + trw['w3']*eps)
    den = sum([trw[f'w{i}'] for i in range(1,4)])
    expected_return = nom/den

    expected_price = M[-1] * np.exp(expected_return)
    order_price = np.random.uniform(expected_price-parameters['P_d'], expected_price+parameters['P_d'])
    
    if order_price >= expected_price:
        side = 'sell'
        order_price = int(np.ceil(order_price))
    else:
        side = 'buy'
        order_price = int(np.floor(order_price))

    return order_price, side

def Run(parameters, seed):
    np.random.seed(seed)
    
    # Initiate the LOB
    bids, asks = warm_LOB(parameters)

    traders_wieghts = assign_traders_weights(parameters)
    

    M = []  # mid price path
    M.append(mid_price(bids, asks))  # M0

    # Start the simulation
    for t in range(1, parameters['T_E']+1):
        
        asks, bids = remove_exp_orders(asks, bids, t, parameters)
        bids, asks = execute_match(bids, asks)
        trw = traders_wieghts[(t-1) % parameters['N']]
        
        order_price, side = traders_price(parameters, trw, M)
        new_order = {"side": side, "price": order_price}

        bids, asks, traded= place_and_match(new_order, t, bids, asks)
        M.append(mid_price(bids, asks))

    M = M[parameters['B']:] # Burn-in 
    M = np.array(M, dtype=float)
    M = forward_fill(M) # Keep the previous value if current value is nan(Bids or Asks is empty)
    return M


In [ ]:
# Define parameters
parameters2 = {
    'N' : 500,
    'w1_max': 1,
    'w2_max': 10,
    'w3_max': 1,
    'P_f' : 10_000,
    'tau_max' : 10_000,
    'sigma_noise': 0.03,
    'P_d': 1_000,
    't_c' : 10_000,
    'T_E' : 100_000,
    'L' : 200,
    'delta' : 1,
    'D' : 5,
    'R' : 30
}
parameters2['B'] = int(0.01 * parameters2['T_E'])

metrics2 = simulation_parallel(parameters2, n_workers=32)

with open(f"metrics_2.pkl", "wb") as f:
    pkl.dump(metrics2, f)

In [3]:
with open("metrics_2.pkl", "rb") as f:
    metrics2 = pkl.load(f)
print(f'BT Metrics:\n{metrics2}')


BT Metrics:
{'acf_lag1': np.float64(0.12180690572103357), 'acf_lag2': np.float64(0.07962394858344335), 'acf_lag3': np.float64(0.06027594475735306), 'acf_lag4': np.float64(0.06218427933047384), 'acf_lag5': np.float64(0.04380493686604721), 'acf_lag6': np.float64(0.0465476062707915), 'Excess_kurtosis': np.float64(3.8561841955221814)}


# S&P500 Benchamrk

In [4]:
sp500 = pd.read_excel('sp500_data.xlsx')
sp500 = sp500.sort_values(by='date').reset_index(drop=True)
sp500['daily log return'] = np.log(sp500['close']/sp500['close'].shift(1))
sp500 = sp500[sp500['date']>= '2013-01-01']
sp500 = sp500[sp500['date']<= '2018-01-01']
sp500 = sp500.reset_index(drop=True)
# Calcualte excess kurtosis
excess_kurt_biased = kurtosis(sp500['daily log return'], bias=True)
excess_kurt_unbiased = kurtosis(sp500['daily log return'], bias=False)

# Calcualte acf
max_lag = 6
acf_sq_vals = acf(sp500['daily log return']**2, nlags=max_lag)

metrics3 = {f'acf_lag{i}': j for i,j in enumerate(acf_sq_vals) if i!=0}
metrics3['Excess_kurtosis'] =  excess_kurt_unbiased

for i in range(1,7):
    print(f'mean ACF of squared returns at lag {i} = {metrics3[f"acf_lag{i}"]}\n')

print(f'mean excess kurtosis of returns = {metrics3["Excess_kurtosis"]}')


mean ACF of squared returns at lag 1 = 0.2959819690178484

mean ACF of squared returns at lag 2 = 0.27755705484469523

mean ACF of squared returns at lag 3 = 0.23801176238012425

mean ACF of squared returns at lag 4 = 0.2385633462014095

mean ACF of squared returns at lag 5 = 0.11612698741609782

mean ACF of squared returns at lag 6 = 0.15178611785621482

mean excess kurtosis of returns = 2.9084244551534777


# Comparison

In [18]:
df = pd.concat([pd.DataFrame([metrics1]), pd.DataFrame([metrics2]), pd.DataFrame([metrics3])])
df.index = ['ZIT', 'BT', 'S&P500']
df = df.T.round(2)
print(df)

                  ZIT    BT  S&P500
acf_lag1         0.20  0.12    0.30
acf_lag2         0.08  0.08    0.28
acf_lag3         0.07  0.06    0.24
acf_lag4         0.07  0.06    0.24
acf_lag5         0.08  0.04    0.12
acf_lag6         0.05  0.05    0.15
Excess_kurtosis  1.04  3.86    2.91
